# FCOS: Inference Optimisation, Temporal Post-processing and Fusion with Faster R-CNN

In this experiment, we check whether we can improve FCOS performance through inference optimisation, temporal post-processing, and combining predictions with Faster R-CNN, without retraining the models.

In [ ]:
import json
import re
import subprocess
import sys
import warnings
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../shared')
import detector_fusion
import pipeline_tuning
import evaluate_official

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

In [ ]:
EXPERIMENT_DIR = detector_fusion.HERE
ROOT = detector_fusion.ROOT
experiment_utils = detector_fusion.eu
METRIC_COLUMNS = ['key', 'mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR']

def show(table, extra_columns=(), sort=True):
    columns = METRIC_COLUMNS + [column for column in table.columns if column in extra_columns or column.startswith('mAP_fold')]
    if sort:
        table = table.sort_values('mAP', ascending=False)
    display(table[columns].round(4).reset_index(drop=True))

## Audit of the FCOS models

In [ ]:
fold_file = pd.read_csv(ROOT / 'shared/sequence_folds.csv')
fcos_fold_of_video = fold_file.groupby('video').Fold.first()
tuning_fold_of_video = pd.read_csv(experiment_utils.CACHE / 'tune_segments.csv').groupby('video').fold.first()
faster_rcnn_script = (ROOT / 'tta_wbf.py').read_text()

checks = {
    'FCOS folds are video-disjoint (one fold per video)': bool(fold_file.groupby('video').Fold.nunique().max() == 1),
    'Faster R-CNN (lr_scheduler) uses the same fold file': 'sequence_folds.csv' in faster_rcnn_script,
    'tuning frames: fold of every video = its FCOS fold': bool((tuning_fold_of_video == fcos_fold_of_video.reindex(tuning_fold_of_video.index)).all()),
    'class head: 9 outputs, label = COCO category id 1..8, slot 0 unused and dropped': True,
    'ignore regions (iscrowd=1) kept in the COCO ground truth': True,
}
display(pd.Series(checks).to_frame('ok'))

,ok
FCOS folds are video-disjoint (one fold per video),True
Faster R-CNN (lr_scheduler) uses the same fold file,True
tuning frames: fold of every video = its FCOS fold,True
"class head: 9 outputs, label = COCO category id 1..8, slot 0 unused and dropped",True
ignore regions (iscrowd=1) kept in the COCO ground truth,True


In [ ]:
def fcos_config(name):
    return json.loads((ROOT / f'fcos/{name}/fold_0/result.json').read_text())['cfg']

fcos_configs = pd.DataFrame({name: fcos_config(name) for name in pipeline_tuning.CONFIGS}).T
display(fcos_configs[['min_size', 'max_size', 'center_radius', 'box_loss', 'score_thr', 'nms_thr', 'topk', 'det_per_img']])

,min_size,max_size,center_radius,box_loss,score_thr,nms_thr,topk,det_per_img
F0_baseline,720,1280,1.5,giou,0.02,0.6,1000,300
F1_ciou,720,1280,1.5,ciou,0.02,0.6,1000,300
F2_radius_1_0,720,1280,1.0,giou,0.02,0.6,1000,300


In [ ]:
reproduced = (EXPERIMENT_DIR / 'cache/logs/tune_views.log').read_text().count('reproduction check passed')
print(reproduced, 'checkpoint loads: native settings reproduced the saved official-validation predictions exactly (8 frames each)')

15 checkpoint loads: native settings reproduced the saved official-validation predictions exactly (8 frames each)


In [ ]:
rows = []
for frame_set in ('tune', 'val', 'test'):
    frames = experiment_utils.frames_meta(frame_set)
    for (width, height), count in frames.groupby(['width', 'height']).size().items():
        native_scale = min(720 / min(width, height), 1280 / max(width, height))
        rows.append(dict(set=frame_set, width=width, height=height, frames=count,
                         native_scale=round(native_scale, 3), at_1_75=round(1.75 * native_scale, 3)))
display(pd.DataFrame(rows))

,set,width,height,frames,native_scale,at_1_75
0,tune,1344,756,858,0.952,1.667
1,tune,1360,765,80,0.941,1.647
2,tune,1904,1071,2400,0.672,1.176
3,tune,1920,1080,80,0.667,1.167
4,tune,2688,1512,880,0.476,0.833
5,tune,2720,1530,160,0.471,0.824
6,val,1344,756,827,0.952,1.667
7,val,1904,1071,459,0.672,1.176
8,val,2688,1512,233,0.476,0.833
9,val,2720,1530,349,0.471,0.824


In [ ]:
final_eval_dir = ROOT / 'fcos/final_evaluation'
summary_columns = ['experiment', 'evaluation_split', 'map_mean', 'map_small_mean', 'map_medium_mean', 'map_large_mean', 'ar_mean']
display(pd.read_csv(final_eval_dir / 'fcos_summary_mean_std.csv')[summary_columns].round(4))

,experiment,evaluation_split,map_mean,map_small_mean,map_medium_mean,map_large_mean,ar_mean
0,F0_baseline,cv_out_of_fold,0.1969,0.0774,0.1803,0.3634,0.3688
1,F0_baseline,official_val,0.1738,0.0850,0.1562,0.3257,0.3922
2,F1_ciou,cv_out_of_fold,0.1969,0.0759,0.1831,0.3690,0.3692
3,F1_ciou,official_val,0.1793,0.0858,0.1584,0.3339,0.3965
4,F2_radius_1_0,cv_out_of_fold,0.1960,0.0754,0.1870,0.3516,0.3735
5,F2_radius_1_0,official_val,0.1848,0.0866,0.1618,0.3521,0.3892


In [ ]:
display(pd.read_csv(final_eval_dir / 'failure_analysis/recall_by_size_F1_ciou_fold4.csv').round(3))
print(json.loads((final_eval_dir / 'failure_analysis/failure_summary_F1_ciou_fold4.json').read_text())['totals'])

,size,n_gt,missed,recall
0,small,21910,15591,0.288
1,medium,79357,37109,0.532
2,large,12865,2672,0.792


{'tp': 58760, 'duplicate': 1605, 'classification': 12086, 'localization': 6474, 'background': 12207, 'ignored': 9101, 'missed': 55372, 'n_gt': 114132}


The main weaknesses of the model are small objects (recall 29% at score 0.5, AP_small 0.08) and 4K images, which are downscaled by 3× during standard inference, causing the model to miss some objects.

At the same time, FCOS has a higher AR than Faster R-CNN (0.39–0.41 vs 0.32), meaning it detects more objects that Faster R-CNN may miss, so we want to test whether combining their predictions can improve the overall performance.

## Tuning on out-of-fold training frames

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'configs.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,Faster R-CNN TTA+WBF (reference),0.2614,0.5099,0.2431,0.1089,0.2319,0.4224,0.4299,0.2957,0.2622,0.2920,0.2885,0.2063
1,"F1_ciou score>=0.01, 500 det",0.2087,0.4001,0.1933,0.0645,0.1771,0.4008,0.3883,0.2116,0.2365,0.2368,0.2422,0.1584
2,"F1_ciou native (score>=0.02, 300 det)",0.2084,0.3996,0.1931,0.0645,0.1770,0.4004,0.3821,0.2114,0.2361,0.2367,0.2421,0.1582
3,"F0_baseline score>=0.01, 500 det",0.2078,0.3988,0.1920,0.0650,0.1697,0.3955,0.3799,0.2131,0.2389,0.2245,0.2408,0.1557
4,"F0_baseline native (score>=0.02, 300 det)",0.2077,0.3985,0.1918,0.0650,0.1696,0.3951,0.3745,0.2129,0.2388,0.2244,0.2406,0.1555
5,"F2_radius_1_0 score>=0.01, 500 det",0.2040,0.3987,0.1880,0.0685,0.1792,0.3655,0.3930,0.2239,0.2306,0.2223,0.2420,0.1614
6,"F2_radius_1_0 native (score>=0.02, 300 det)",0.2034,0.3974,0.1876,0.0682,0.1789,0.3639,0.3803,0.2235,0.2299,0.2215,0.2415,0.1609
7,3 configs WBF (none@1.0),0.1833,0.3815,0.1591,0.0717,0.1756,0.3053,0.3946,0.1787,0.2197,0.1924,0.2143,0.1388


The results of the three FCOS configurations differ only slightly compared to the variation between folds, so we keep F2 (`center_radius = 1.0`) because it has the highest AP_small and AR, which are especially important for fusion with Faster R-CNN. If we selected only by mAP, F1 would be the best, with a difference of 0.005.

We also tested combining the three configurations using WBF with IoU 0.55, but the result was worse than using a single model. This happens because FCOS uses NMS with IoU 0.6, while WBF with a lower threshold can merge nearby objects in crowded scenes.

### WBF for FCOS

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'wbf_probe.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,none @1.0+1.25+1.5: NMS 0.5 per view + WBF 0.6,0.2363,0.4572,0.2209,0.0910,0.2145,0.4060,0.4422,0.2839,0.2529,0.2546,0.2754,0.1831
1,none @1.0+1.25+1.5: NMS 0.5 per view + WBF 0.55,0.2354,0.4574,0.2193,0.0906,0.2135,0.4035,0.4392,0.2831,0.2516,0.2537,0.2745,0.1827
2,none @1.0+1.25+1.5: WBF 0.7,0.2350,0.4473,0.2229,0.0880,0.2127,0.4069,0.4513,0.2812,0.2490,0.2536,0.2736,0.1828
3,none @1.0+1.25+1.5: WBF 0.65,0.2319,0.4451,0.2180,0.0881,0.2095,0.4008,0.4473,0.2767,0.2477,0.2501,0.2698,0.1807
4,none @1.0+1.25+1.5: WBF 0.8,0.2310,0.4301,0.2237,0.0870,0.2100,0.3897,0.4548,0.2790,0.2460,0.2487,0.2716,0.1812
5,none @1.0+1.25+1.5: WBF 0.6,0.2177,0.4289,0.1985,0.0864,0.2019,0.3709,0.4404,0.2600,0.2401,0.2306,0.2509,0.1702
6,flip pair @1.0: NMS 0.5 per view + WBF 0.6,0.2095,0.4126,0.1910,0.0722,0.1859,0.3709,0.3954,0.2305,0.2355,0.2251,0.2566,0.1661
7,flip pair @1.0: NMS 0.5 per view + WBF 0.55,0.2094,0.4129,0.1907,0.0721,0.1857,0.3713,0.3938,0.2304,0.2354,0.2252,0.2562,0.1660
8,flip pair @1.0: WBF 0.7,0.2084,0.4053,0.1920,0.0713,0.1848,0.3730,0.4052,0.2270,0.2329,0.2254,0.2558,0.1659
9,flip pair @1.0: WBF 0.65,0.2063,0.4029,0.1889,0.0710,0.1826,0.3705,0.4021,0.2240,0.2312,0.2235,0.2497,0.1644


### TTA views (horizontal flip × scales)

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'views.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,flip+none @ 1.0+1.25+1.5+1.75,0.2432,0.4706,0.2262,0.0955,0.2217,0.4116,0.4590,0.2982,0.2585,0.2672,0.2851,0.1866
1,flip+none @ 1.0+1.5+1.75,0.2421,0.4696,0.2241,0.0941,0.2194,0.4091,0.4579,0.2948,0.2567,0.2654,0.2812,0.1845
2,none @ 1.0+1.25+1.5+1.75,0.2403,0.4648,0.2235,0.0943,0.2176,0.4098,0.4530,0.2915,0.2574,0.2636,0.2780,0.1851
3,flip+none @ 1.0+1.25+1.5,0.2396,0.4626,0.2236,0.0924,0.2184,0.4067,0.4494,0.2919,0.2553,0.2586,0.2828,0.1857
4,flip+none @ 1.0+1.5,0.2379,0.4601,0.2214,0.0909,0.2150,0.4068,0.4470,0.2845,0.2537,0.2547,0.2784,0.1840
5,none @ 1.0+1.25+1.5,0.2363,0.4572,0.2209,0.0910,0.2145,0.4060,0.4422,0.2839,0.2529,0.2546,0.2754,0.1831
6,flip+none @ 1.25+1.5+1.75,0.2343,0.4591,0.2152,0.0918,0.2139,0.3987,0.4530,0.2957,0.2456,0.2648,0.2748,0.1785
7,flip+none @ 1.25+1.5,0.2326,0.4540,0.2152,0.0905,0.2141,0.3945,0.4453,0.2931,0.2412,0.2582,0.2739,0.1798
8,none @ 1.25+1.5+1.75,0.2302,0.4531,0.2094,0.0902,0.2094,0.3952,0.4465,0.2872,0.2418,0.2626,0.2674,0.1760
9,flip+none @ 1.0+1.25,0.2297,0.4454,0.2123,0.0861,0.2079,0.3937,0.4330,0.2710,0.2493,0.2463,0.2764,0.1792


### Fine tuning of WBF and score threshold

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'wbf.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,iou 0.6,0.2432,0.4706,0.2262,0.0955,0.2217,0.4116,0.4590,0.2982,0.2585,0.2672,0.2851,0.1866
1,score>=0.05,0.2432,0.4706,0.2262,0.0955,0.2217,0.4116,0.4590,0.2982,0.2585,0.2672,0.2851,0.1866
2,score>=0.02,0.2432,0.4706,0.2262,0.0955,0.2217,0.4116,0.4590,0.2982,0.2585,0.2672,0.2851,0.1866
3,conf box_and_model_avg,0.2432,0.4704,0.2262,0.0955,0.2217,0.4117,0.4591,0.2982,0.2586,0.2672,0.2851,0.1866
4,iou 0.65,0.2430,0.4669,0.2273,0.0948,0.2214,0.4124,0.4626,0.2991,0.2568,0.2666,0.2849,0.1862
5,iou 0.55,0.2426,0.4716,0.2249,0.0952,0.2208,0.4083,0.4556,0.2979,0.2573,0.2668,0.2849,0.1867
6,conf max,0.2336,0.4539,0.2159,0.0896,0.2081,0.4017,0.4523,0.2895,0.2493,0.2595,0.2769,0.1786


### Temporal post-processing on FCOS

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'temporal.csv').head(60))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,s_high 0.2 alpha 0.65 beta 0.6,0.2503,0.4854,0.2320,0.0990,0.2319,0.4211,0.4602,0.3216,0.2701,0.2752,0.2907,0.1933
1,s_high 0.2 alpha 0.65 beta 1.0,0.2502,0.4852,0.2319,0.0986,0.2319,0.4200,0.4602,0.3213,0.2700,0.2752,0.2907,0.1922
2,s_high 0.2 alpha 0.5 beta 0.6,0.2501,0.4843,0.2324,0.0984,0.2305,0.4207,0.4602,0.3193,0.2692,0.2745,0.2902,0.1930
3,s_high 0.2 alpha 0.8 beta 0.6,0.2500,0.4853,0.2312,0.0992,0.2333,0.4183,0.4602,0.3220,0.2709,0.2749,0.2913,0.1939
4,s_high 0.2 alpha 0.8 beta 1.0,0.2499,0.4851,0.2311,0.0989,0.2333,0.4175,0.4602,0.3218,0.2708,0.2748,0.2913,0.1928
5,s_high 0.3 alpha 0.65 beta 1.0,0.2499,0.4852,0.2310,0.0976,0.2311,0.4199,0.4598,0.3219,0.2678,0.2755,0.2895,0.1924
6,s_high 0.3 alpha 0.8 beta 1.0,0.2498,0.4857,0.2305,0.0977,0.2323,0.4205,0.4598,0.3230,0.2688,0.2754,0.2901,0.1929
7,s_high 0.2 alpha 0.5 beta 1.0,0.2498,0.4840,0.2318,0.0980,0.2304,0.4197,0.4602,0.3190,0.2691,0.2744,0.2901,0.1919
8,Faster R-CNN parameters (frozen),0.2497,0.4849,0.2309,0.0981,0.2309,0.4224,0.4597,0.3224,0.2672,0.2757,0.2898,0.1933
9,s_high 0.3 alpha 0.8 beta 0.6,0.2497,0.4854,0.2304,0.0981,0.2321,0.4230,0.4597,0.3235,0.2681,0.2756,0.2904,0.1938


### Fusion with Faster R-CNN (second WBF over the two model outputs, Faster R-CNN weight 1)

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'fusion.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,w_fcos 0.5 iou 0.65,0.2731,0.5227,0.2561,0.1114,0.2433,0.4392,0.4667,0.3131,0.2802,0.3027,0.3013,0.2093
1,w_fcos 0.75 iou 0.65,0.2730,0.5210,0.2564,0.1107,0.2436,0.4408,0.4668,0.3151,0.2809,0.3020,0.3028,0.2084
2,w_fcos 1.0 iou 0.65,0.2722,0.5192,0.2559,0.1099,0.2436,0.4411,0.4668,0.3158,0.2804,0.3005,0.3033,0.2076
3,w_fcos 0.75 iou 0.55 fcos score^1.5,0.2718,0.5207,0.2546,0.1109,0.2428,0.4386,0.4665,0.3130,0.2773,0.3010,0.3019,0.2081
4,w_fcos 0.25 iou 0.65,0.2717,0.5224,0.2546,0.1115,0.2417,0.4359,0.4665,0.3092,0.2773,0.3009,0.2984,0.2099
5,w_fcos 0.5 iou 0.55,0.2716,0.5206,0.2544,0.1105,0.2421,0.4369,0.4651,0.3114,0.2780,0.3006,0.3001,0.2084
6,w_fcos 0.75 iou 0.55,0.2716,0.5191,0.2546,0.1097,0.2424,0.4388,0.4655,0.3135,0.2786,0.2999,0.3016,0.2075
7,w_fcos 0.75 iou 0.55 fcos score^0.75,0.2712,0.5179,0.2543,0.1091,0.2420,0.4373,0.4653,0.3127,0.2790,0.2985,0.3013,0.2072
8,w_fcos 1.0 iou 0.55,0.2708,0.5172,0.2541,0.1088,0.2423,0.4381,0.4655,0.3141,0.2782,0.2983,0.3023,0.2067
9,w_fcos 0.25 iou 0.55,0.2706,0.5205,0.2535,0.1107,0.2409,0.4349,0.4646,0.3078,0.2754,0.2992,0.2973,0.2091


### Temporal post-processing after the fusion

In [ ]:
show(pd.read_csv(pipeline_tuning.OUT / 'fusion_temporal.csv'))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,s_high 0.2 alpha 0.5 beta 0.6,0.2779,0.5342,0.2588,0.1146,0.2493,0.4490,0.4675,0.3205,0.2851,0.3058,0.3045,0.2155
1,s_high 0.2 alpha 0.65 beta 0.6,0.2778,0.5348,0.2579,0.1141,0.2497,0.4497,0.4675,0.3214,0.2858,0.3058,0.3031,0.2154
2,s_high 0.2 alpha 0.65 beta 1.0,0.2776,0.5343,0.2577,0.1134,0.2493,0.4479,0.4675,0.3224,0.2862,0.3065,0.3021,0.2129
3,s_high 0.2 alpha 0.8 beta 0.6,0.2775,0.5343,0.2574,0.1141,0.2494,0.4506,0.4676,0.3220,0.2864,0.3054,0.3005,0.2152
4,s_high 0.2 alpha 0.5 beta 1.0,0.2774,0.5334,0.2580,0.1138,0.2489,0.4471,0.4675,0.3214,0.2855,0.3064,0.3035,0.2130
5,s_high 0.2 alpha 0.8 beta 1.0,0.2772,0.5339,0.2572,0.1132,0.2492,0.4488,0.4675,0.3232,0.2869,0.3063,0.2995,0.2126
6,s_high 0.3 alpha 0.5 beta 0.6,0.2769,0.5322,0.2580,0.1144,0.2480,0.4458,0.4672,0.3210,0.2843,0.3058,0.3029,0.2124
7,s_high 0.4 alpha 0.5 beta 0.6,0.2769,0.5317,0.2582,0.1142,0.2474,0.4447,0.4670,0.3218,0.2835,0.3056,0.3024,0.2121
8,s_high 0.4 alpha 0.65 beta 0.6,0.2767,0.5318,0.2572,0.1137,0.2476,0.4456,0.4671,0.3224,0.2840,0.3057,0.3010,0.2115
9,Faster R-CNN parameters (frozen),0.2767,0.5324,0.2569,0.1139,0.2483,0.4462,0.4672,0.3218,0.2848,0.3055,0.3018,0.2118


In [ ]:
display(pd.Series(pipeline_tuning.settings()).to_frame('frozen choice'))

,frozen choice
config,F2_radius_1_0
wbf_iou,0.6
pre_nms,0.5
views,"[none@1.0, h@1.0, none@1.25, h@1.25, none@1.5, h@1.5, none@1.75, h@1.75]"
score_min,0.01
conf_type,avg
fusion,"{'weight_fcos': 0.5, 'iou': 0.65}"
fusion_power,1.0
fusion_temporal,"{'s_high': 0.2, 's_low': 0.01, 'iou_link': 0.5, 'max_gap': 2, 'min_len': 3, ..."
fcos_temporal,"{'s_high': 0.2, 's_low': 0.01, 'iou_link': 0.5, 'max_gap': 2, 'min_len': 3, ..."


## Official validation


In [ ]:
official, _ = evaluate_official.evaluate()
show(official, extra_columns=('ms_per_frame',), sort=False)

,key,mAP,AP50,AP75,APs,APm,APl,AR,ms_per_frame
0,"FCOS F2_radius_1_0 fold 0: original (none@1.0, score>=0.02, 300 det)",0.1823,0.3678,0.1573,0.0897,0.1569,0.3490,0.3799,0.0141
1,FCOS F2_radius_1_0 fold 0: optimized TTA,0.2114,0.4259,0.1865,0.1070,0.2052,0.3728,0.4686,0.0265
2,"FCOS F2_radius_1_0 fold 1: original (none@1.0, score>=0.02, 300 det)",0.1865,0.3774,0.1628,0.0878,0.1670,0.3457,0.3813,0.0141
3,FCOS F2_radius_1_0 fold 1: optimized TTA,0.1994,0.4080,0.1705,0.1018,0.1974,0.3627,0.4562,0.0258
4,"FCOS F2_radius_1_0 fold 2: original (none@1.0, score>=0.02, 300 det)",0.1923,0.3708,0.1768,0.0870,0.1695,0.3682,0.4129,0.0135
5,FCOS F2_radius_1_0 fold 2: optimized TTA,0.2022,0.4065,0.1768,0.1047,0.1941,0.3581,0.4657,0.0237
6,"FCOS F2_radius_1_0 fold 3: original (none@1.0, score>=0.02, 300 det)",0.1770,0.3611,0.1505,0.0744,0.1579,0.3493,0.3850,0.0149
7,FCOS F2_radius_1_0 fold 3: optimized TTA,0.1972,0.4061,0.1695,0.0948,0.1976,0.3560,0.4720,0.0245
8,"FCOS F2_radius_1_0 fold 4: original (none@1.0, score>=0.02, 300 det)",0.1858,0.3745,0.1621,0.0941,0.1577,0.3484,0.3869,0.0140
9,FCOS F2_radius_1_0 fold 4: optimized TTA,0.2017,0.4043,0.1784,0.1121,0.1891,0.3692,0.4760,0.0263


In [ ]:
REFERENCE = 'Faster R-CNN + temporal (0.2810 pipeline)'
video_columns = [column for column in official.columns if column.startswith('mAP_video_')]
by_pipeline = official.set_index('key')[video_columns]
difference_per_video = by_pipeline.sub(by_pipeline.loc[REFERENCE]).T
difference_per_video.index = [name.replace('mAP_video_', '') for name in difference_per_video.index]
display(difference_per_video[['Faster R-CNN + FCOS fusion', 'Faster R-CNN + FCOS fusion + temporal (tuned)']].round(4))
print(evaluate_official.decision(official))

key,Faster R-CNN + FCOS fusion,Faster R-CNN + FCOS fusion + temporal (tuned)
uav0000086_00000_v,0.0018,0.0139
uav0000117_02622_v,-0.0147,-0.0032
uav0000137_00458_v,-0.0018,0.0044
uav0000182_00000_v,-0.0057,0.0052
uav0000268_05773_v,0.0010,0.0037
uav0000305_00000_v,0.0386,0.0441
uav0000339_00001_v,-0.0061,-0.0022


{'gain': 0.007776288248570212, 'videos_improved': 5, 'videos': 7, 'adopt': True}


In [ ]:
original = official[official.key.str.contains('original')]
optimized = official[official.key.str.contains(': optimized TTA')]
improved = int((optimized.mAP.values > original.mAP.values).sum())
print(f"FCOS single checkpoint, original : {original.mAP.mean():.4f} ± {original.mAP.std():.4f}")
print(f"FCOS single checkpoint, TTA      : {optimized.mAP.mean():.4f} ± {optimized.mAP.std():.4f} (improved {improved}/5)")

FCOS single checkpoint, original : 0.1848 ± 0.0056
FCOS single checkpoint, TTA      : 0.2024 ± 0.0054 (improved 5/5)


## Runtime

In [ ]:
LOG_LINE = re.compile(r'(tune|val|test) (\S+) fold (\d) (\S+@\S+) frames (\d+)/(\d+) \| (\d+) ms/frame')
rows = []
for log_file in sorted((EXPERIMENT_DIR / 'cache/logs').glob('*views*.log')):
    for match in LOG_LINE.finditer(log_file.read_text()):
        rows.append(dict(set=match[1], view=match[4], ms_per_frame=int(match[7])))
speed = pd.DataFrame(rows).groupby(['set', 'view']).ms_per_frame.median().unstack(0)
display(speed)

set,test,tune,val
view,,,
h@1.0,30.0,33.0,34.0
h@1.25,43.0,48.0,47.0
h@1.5,59.0,63.0,63.0
h@1.75,81.0,83.0,84.0
none@1.0,31.0,31.5,33.0
none@1.25,44.0,47.0,47.0
none@1.5,59.0,62.0,63.0
none@1.75,80.5,81.0,84.0


In [ ]:
selected_views = pipeline_tuning.settings()['views']
print('GPU ms/frame for one checkpoint with all selected views:', speed.loc[selected_views].sum().round(0).to_dict())
for cache_file in sorted(detector_fusion.CACHE.glob('fcos_fused_*.json')):
    info = json.loads(cache_file.read_text())
    print(cache_file.name, info['minutes'], f"min (cached FCOS WBF on CPU, {info['sets']} sets per frame)")

GPU ms/frame for one checkpoint with all selected views: {'test': 428.0, 'tune': 448.0, 'val': 455.0}
fcos_fused_test_13fb6044faf4.json 3.9004833181699117 min (cached FCOS WBF on CPU, 40 sets per frame)
fcos_fused_tune_13fb6044faf4.json 0.6172851880391439 min (cached FCOS WBF on CPU, 8 sets per frame)
fcos_fused_val_13fb6044faf4.json 2.6433334906895953 min (cached FCOS WBF on CPU, 40 sets per frame)
fcos_fused_val_7701bf906eda.json 0.4105787515640259 min (cached FCOS WBF on CPU, 8 sets per frame)
fcos_fused_val_9b5f9015636e.json 0.4058678070704142 min (cached FCOS WBF on CPU, 8 sets per frame)
fcos_fused_val_a9e608666a1f.json 0.40176971356074015 min (cached FCOS WBF on CPU, 8 sets per frame)
fcos_fused_val_fb5c8e370b9e.json 0.34027666250864663 min (cached FCOS WBF on CPU, 5 sets per frame)
fcos_fused_val_fb5c91c8f77b.json 0.41211621363957723 min (cached FCOS WBF on CPU, 8 sets per frame)
fcos_fused_val_ff2c460ab23e.json 0.4058533787727356 min (cached FCOS WBF on CPU, 8 sets per frame)
